# Module 06: Weaviate

## What is a Vector Database?

A traditional database answers queries like: *"Find all creators where cluster_id = 5."* That's exact matching on structured data.

A vector database answers queries like: *"Find the 10 creators most similar to this one."* That requires **approximate nearest-neighbor (ANN) search** — comparing a query vector against millions of stored vectors and returning the closest ones.

The core insight: if you represent data as vectors (embeddings), similarity becomes distance. Finding similar items = finding nearby points in high-dimensional space.

**Why not just use numpy?**
```python
# This works for 20K vectors:
similarities = np.dot(all_embeddings, query_vector)
top_10 = np.argsort(similarities)[-10:]
```
This does exact search, loading all embeddings into RAM. For 20K × 1024-dim vectors: ~80MB, query time ~10ms. Totally fine.

It breaks at scale: 10M vectors × 1024-dim = 40GB RAM, 5 second query time. Weaviate's HNSW index gives you ~5ms queries on 10M vectors using 2GB RAM. That's the value.

**But we use Weaviate even at 20K creators because:**
1. Hybrid search (vector + keyword filters) is hard to build from scratch
2. Weaviate's schema gives us a queryable object store, not just a vector index
3. It's the RAG layer for the chatbot — designed for this use case
4. Learning it at small scale transfers directly to production-scale use

## How HNSW Works (conceptually)

HNSW (Hierarchical Navigable Small World) builds a graph where similar vectors are connected. A query starts at the top layer (few nodes, coarse connections) and navigates down through layers of increasing density, getting closer to the query vector at each step. The result is O(log n) search instead of O(n).

You don't need to understand the internals to use it, but knowing it's a graph explains why: (1) building the index is slower than insertion order matters, and (2) it trades perfect accuracy for speed (approximate nearest neighbors).

## Key Vocabulary

| Term | Meaning | SQL analogy |
|---|---|---|
| **Collection** (formerly Class) | A type of object with a defined schema | Table |
| **Object** | One record in a collection | Row |
| **Properties** | Scalar fields on an object (text, int, etc.) | Columns |
| **Vector** | The embedding attached to an object | No SQL equivalent |
| **Vectorizer** | Module that auto-generates vectors from text | No SQL equivalent |
| **near_vector** | Query: find objects close to a given vector | No SQL equivalent |
| **near_text** | Query: embed text, then find similar objects | Full-text search |
| **BM25** | Keyword-based text search | LIKE / full-text index |
| **Hybrid** | Combines vector + BM25 results | N/A |


## Setup

We use **Weaviate Embedded** — Weaviate runs inside your Python process, no Docker needed.

```bash
pip install weaviate-client sentence-transformers numpy
```

Or in production/Galaxy: Weaviate runs as a Helm chart in k3s, and you connect with:
```python
client = weaviate.connect_to_local(host="weaviate-service", port=8080)
```

In [ ]:
import weaviate
from weaviate.embedded import EmbeddedOptions
import numpy as np

# Start embedded Weaviate (runs locally, no Docker)
client = weaviate.connect_to_embedded(
    version="1.24.4",
    options=EmbeddedOptions(
        persistence_data_path="./weaviate_data",  # data persists between runs
    )
)

print(f"Weaviate ready: {client.is_ready()}")
print(f"Version: {client.get_meta()['version']}")

---
## Exercise 1: Create a Collection

Collections define the schema — what properties objects have and how they're vectorized. We use `vectorizer='none'` because we supply our own pre-computed embeddings (from GTE-Large).

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

# Delete if it already exists (for re-running this notebook)
if client.collections.exists("Creator"):
    client.collections.delete("Creator")

# Create the Creator collection
client.collections.create(
    name="Creator",
    vectorizer_config=Configure.Vectorizer.none(),  # We supply our own vectors
    properties=[
        Property(name="creator_id", data_type=DataType.TEXT),
        Property(name="display_name", data_type=DataType.TEXT),
        Property(name="bio_text", data_type=DataType.TEXT),
        Property(name="cluster_id", data_type=DataType.INT),
        Property(name="cluster_name", data_type=DataType.TEXT),
        Property(name="subscriber_count", data_type=DataType.INT),
    ]
)

print("Creator collection created")
print(f"Collections: {[c.name for c in client.collections.list_all().values()]}")

---
## Exercise 2: Insert Objects with Custom Vectors

In the Galaxy project, vectors come from GTE-Large-v1.5 (1024-dim). Here we use small random vectors to keep things fast. The API is identical.

In [ ]:
# Simulated creator data (in Galaxy, this comes from the Parquet produced by training)
creators = [
    {"id": "fandom_001", "name": "MrBeast", "bio": "I give away money and do extreme challenges", "cluster": 0, "cluster_name": "Challenge & Philanthropy", "subs": 250_000_000},
    {"id": "fandom_002", "name": "PewDiePie", "bio": "Gaming commentary and meme review", "cluster": 1, "cluster_name": "Gaming Commentary", "subs": 111_000_000},
    {"id": "fandom_003", "name": "Markiplier", "bio": "Horror game playthroughs and charity streams", "cluster": 1, "cluster_name": "Gaming Commentary", "subs": 35_000_000},
    {"id": "fandom_004", "name": "Kurzgesagt", "bio": "Science and philosophy explained with animation", "cluster": 2, "cluster_name": "Educational Animation", "subs": 22_000_000},
    {"id": "fandom_005", "name": "CGP Grey", "bio": "Explainer videos on history, politics, and systems", "cluster": 2, "cluster_name": "Educational Animation", "subs": 5_000_000},
    {"id": "fandom_006", "name": "Dream", "bio": "Minecraft speedruns, manhunts, and challenges", "cluster": 3, "cluster_name": "Minecraft Competitive", "subs": 35_000_000},
    {"id": "fandom_007", "name": "Technoblade", "bio": "Minecraft PvP, lore, and pig-related content", "cluster": 3, "cluster_name": "Minecraft Competitive", "subs": 14_000_000},
    {"id": "fandom_008", "name": "Linus Tech Tips", "bio": "PC hardware reviews and tech news", "cluster": 4, "cluster_name": "Tech Hardware", "subs": 15_000_000},
]

# Generate a fake embedding per creator
# In Galaxy: these come from GTE-Large inference on creator bios
np.random.seed(42)
# Creators in the same cluster get similar vectors (add cluster signal)
base_vectors = {0: np.random.randn(128), 1: np.random.randn(128), 
                2: np.random.randn(128), 3: np.random.randn(128), 4: np.random.randn(128)}

def fake_embed(creator):
    base = base_vectors[creator["cluster"]]
    noise = np.random.randn(128) * 0.3  # small noise within cluster
    vec = base + noise
    return (vec / np.linalg.norm(vec)).tolist()  # normalize

# Batch insert
collection = client.collections.get("Creator")

with collection.batch.dynamic() as batch:
    for c in creators:
        batch.add_object(
            properties={
                "creator_id": c["id"],
                "display_name": c["name"],
                "bio_text": c["bio"],
                "cluster_id": c["cluster"],
                "cluster_name": c["cluster_name"],
                "subscriber_count": c["subs"],
            },
            vector=fake_embed(c)
        )

count = collection.aggregate.over_all(total_count=True).total_count
print(f"Inserted {count} creators")

---
## Exercise 3: Vector Similarity Search

`near_vector` finds the objects closest to a given vector. This is the core operation that powers "find creators similar to X."

In [ ]:
# Get MrBeast's vector to use as a query
# In Galaxy: the chatbot retrieves this from Feast (Redis cache) or re-embeds the query
results = collection.query.fetch_objects(
    filters=weaviate.classes.query.Filter.by_property("creator_id").equal("fandom_001"),
    include_vector=True
)
mrbeast_vector = results.objects[0].vector["default"]
print(f"MrBeast vector shape: {len(mrbeast_vector)}")

In [ ]:
# Find creators similar to MrBeast
from weaviate.classes.query import MetadataQuery

response = collection.query.near_vector(
    near_vector=mrbeast_vector,
    limit=5,
    return_metadata=MetadataQuery(distance=True),  # include similarity score
    return_properties=["display_name", "cluster_name", "bio_text"]
)

print("Creators most similar to MrBeast:")
for obj in response.objects:
    dist = obj.metadata.distance
    print(f"  [{1-dist:.3f} similarity] {obj.properties['display_name']} "
          f"({obj.properties['cluster_name']})")
    print(f"    Bio: {obj.properties['bio_text'][:80]}...")

---
## Exercise 4: Filtered Vector Search

Combine vector similarity with scalar filters. Example: "Find creators similar to MrBeast **but only within the gaming cluster**" or "Find creators similar to this embedding **with more than 10M subscribers**."

In [ ]:
from weaviate.classes.query import Filter

# Find creators similar to MrBeast, but only gaming creators
response = collection.query.near_vector(
    near_vector=mrbeast_vector,
    limit=5,
    filters=Filter.by_property("cluster_name").equal("Gaming Commentary"),
    return_metadata=MetadataQuery(distance=True),
    return_properties=["display_name", "cluster_name", "subscriber_count"]
)

print("Most similar to MrBeast — Gaming Commentary cluster only:")
for obj in response.objects:
    print(f"  {obj.properties['display_name']} "
          f"({obj.properties['subscriber_count']:,} subs) "
          f"similarity={1-obj.metadata.distance:.3f}")

In [ ]:
# Find creators similar to MrBeast with over 20M subscribers
response = collection.query.near_vector(
    near_vector=mrbeast_vector,
    limit=5,
    filters=Filter.by_property("subscriber_count").greater_than(20_000_000),
    return_metadata=MetadataQuery(distance=True),
    return_properties=["display_name", "cluster_name", "subscriber_count"]
)

print("Most similar to MrBeast — 20M+ subscribers only:")
for obj in response.objects:
    print(f"  {obj.properties['display_name']} "
          f"({obj.properties['subscriber_count']:,} subs)")

---
## Exercise 5: Hybrid Search (Vector + Keyword)

Hybrid search combines vector similarity (semantic) with BM25 keyword matching. Useful when the query contains specific terms that should be weighted heavily (e.g., a creator's name, a niche topic keyword).

The `alpha` parameter controls the blend: `alpha=1.0` is pure vector, `alpha=0.0` is pure BM25, `alpha=0.5` is equal blend.

In [ ]:
# Hybrid: keyword "minecraft" weighted alongside vector similarity
response = collection.query.hybrid(
    query="minecraft gameplay",             # keyword query for BM25
    vector=mrbeast_vector,                  # vector for similarity
    alpha=0.5,                              # 50/50 blend
    limit=5,
    return_metadata=MetadataQuery(score=True, explain_score=True),
    return_properties=["display_name", "cluster_name", "bio_text"]
)

print("Hybrid search (minecraft keyword + MrBeast vector):")
for obj in response.objects:
    print(f"  [{obj.metadata.score:.4f}] {obj.properties['display_name']} "
          f"({obj.properties['cluster_name']})")

---
## Exercise 6: Build a Mini RAG Pipeline

RAG (Retrieval-Augmented Generation): instead of asking an LLM to answer from memory, you retrieve relevant context first and inject it into the prompt. Weaviate handles the retrieval step.

This is the exact pattern the Galaxy chatbot uses.

In [ ]:
def retrieve_context(query_text: str, query_vector: list, top_k: int = 5) -> str:
    """
    Retrieve the most relevant creators for a given query.
    Returns a formatted context string ready to inject into an LLM prompt.
    
    In Galaxy: query_vector comes from GTE-Large embedding of the user's question,
    or from Feast if the query names a specific creator.
    """
    response = collection.query.hybrid(
        query=query_text,
        vector=query_vector,
        alpha=0.6,  # slight preference for vector similarity
        limit=top_k,
        return_properties=["display_name", "bio_text", "cluster_name", "subscriber_count"]
    )

    context_parts = []
    for i, obj in enumerate(response.objects, 1):
        p = obj.properties
        context_parts.append(
            f"{i}. {p['display_name']} — Cluster: {p['cluster_name']} "
            f"({p['subscriber_count']:,} subscribers)\n"
            f"   Bio: {p['bio_text']}"
        )

    return "\n\n".join(context_parts)


def build_rag_prompt(user_question: str, context: str) -> str:
    return f"""You are a YouTube creator analyst. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say so.

Context:
{context}

User question: {user_question}
Answer:"""


# Simulate a user query
user_question = "Which creators make educational science content?"

# In Galaxy: this vector comes from GTE-Large embedding of the question
# Here we use a fake vector biased toward the educational cluster
query_vector = (base_vectors[2] + np.random.randn(128) * 0.1)
query_vector = (query_vector / np.linalg.norm(query_vector)).tolist()

context = retrieve_context(user_question, query_vector)
prompt = build_rag_prompt(user_question, context)

print("=== Retrieved Context ===")
print(context)
print("\n=== Full Prompt to LLM ===")
print(prompt)

In [ ]:
# In Galaxy, this prompt goes to Ollama (local) or Groq (Oracle Cloud)
# Here we show the API call pattern:

print("In Galaxy, this prompt is sent to Ollama:")
print("")
print("import httpx")
print("response = httpx.post(")
print('    "http://ollama-service:11434/api/chat",')
print("    json={")
print('        "model": "mistral:7b-instruct-q8_0",')
print('        "messages": [{"role": "user", "content": prompt}],')
print('        "stream": True')
print("    }")
print(")")

print("")
print("Or to Groq (Oracle Cloud env):")
print("")
print("from groq import Groq")
print("client = Groq(api_key=os.environ['GROQ_API_KEY'])")
print("response = client.chat.completions.create(")
print('    model="llama-3.1-70b-versatile",')
print('    messages=[{"role": "user", "content": prompt}]')
print(")")

In [ ]:
# Clean up embedded Weaviate when done
client.close()
print("Weaviate connection closed")

---
## How Weaviate Maps to the Galaxy Project

```
Pipeline run completes
    ↓
Airflow: WeaviateUpsertTask
    - Reads new embeddings Parquet from Oracle Object Storage
    - For each creator: client.collections.get('Creator').data.insert_many(objects)
    - Uses batch.dynamic() for efficient bulk loading
    ↓
Production Weaviate (k3s on Oracle Cloud)
    - All 20K+ creator objects with 1024-dim GTE vectors
    ↓
FastAPI chatbot service
    1. User asks: "Who makes horror gaming content?"
    2. Embedding service embeds the question → 1024-dim vector
    3. Weaviate hybrid search → top 10 creators
    4. Build prompt with creator cards
    5. Groq API → natural language answer → stream to user
```

The Weaviate client code in production is essentially identical to what you wrote in exercises 2–6 — just connecting to `weaviate.connect_to_local(host='weaviate-service', port=8080)` instead of embedded mode.

## Summary

| Operation | Code |
|---|---|
| Connect (embedded) | `weaviate.connect_to_embedded()` |
| Connect (server) | `weaviate.connect_to_local(host='weaviate-service')` |
| Create collection | `client.collections.create(name, vectorizer_config, properties)` |
| Insert with vector | `collection.data.insert(properties=..., vector=...)` |
| Batch insert | `with collection.batch.dynamic() as batch: batch.add_object(...)` |
| Vector search | `collection.query.near_vector(near_vector=..., limit=...)` |
| Filtered vector search | Add `filters=Filter.by_property(...).equal(...)` |
| Hybrid search | `collection.query.hybrid(query='text', vector=..., alpha=0.5)` |